# Balanced PCA-plane activation rewrite pipeline

This notebook is a configuration-driven wrapper around the reusable Python scripts. Running all cells fits the aggregated three-component PCA and balanced separating plane, saves its artifacts, then rewrites every source activation row using its own plane-projected PCA score.

In [1]:
# CONFIGURATION — edit this cell only.
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'fit_balanced_pca_plane.py').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


ROOT = find_repo_root()

# Input and output paths
INPUT_ACTS_DIR = ROOT / '.acts_filter_1'
REWRITTEN_ACTS_DIR = ROOT / '.acts_filter_2'
ARTIFACT_DIR = ROOT / 'results' / 'balanced_pca_plane'
METADATA_CACHE_DIR = ROOT / 'data' / 'activation_explorer_cache'

# Output artifact filenames (all are written beneath ARTIFACT_DIR)
PCA_FILENAME = 'pca_3_components_filter_2.joblib'
CLASSIFIER_FILENAME = 'balanced_linear_svm_filter_2.joblib'
POINTS_FILENAME = 'original_and_projected_pc_points_filter_2.csv'
PLOT_FILENAME = 'pca_scores_and_plane_filter_2.html'

# Folder-to-class grouping. Values must be the binary labels 0 and 1.
# Dictionary order controls folder discovery and plot legend order.
FOLDER_TO_CLASS = {
    'plain': 0,
    'plain_long': 1,
    'indirect': 1,
    'new_conv': 0,
}

# Raw rows matching ANY listed field/value are ignored during aggregation, PCA fitting,
# and classifier fitting. They are still transformed and written to REWRITTEN_ACTS_DIR.
IGNORE_WHEN_FITTING = {
    'base_unit': ['millennia', 'seconds'],
}

# Model and execution settings
PCA_BATCH_SIZE = 4096
SVM_C = 1.0
RANDOM_STATE = 42
MAX_PLOT_POINTS_PER_FOLDER = 5_000
OVERWRITE_REWRITTEN_BATCHES = True

## 1. Fit PCA and the balanced linear plane

Activations are averaged by `(task, source_folder, time_horizon_months)` before PCA. Classifier sample weights give every configured folder equal total representation.

In [2]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.fit_balanced_pca_plane import fit_balanced_pca_plane
from scripts.rewrite_activations_with_projected_pca import rewrite_activation_batches


artifact_paths = fit_balanced_pca_plane(
    INPUT_ACTS_DIR,
    ARTIFACT_DIR,
    cache_dir=METADATA_CACHE_DIR,
    folders=tuple(FOLDER_TO_CLASS),
    class_by_folder=FOLDER_TO_CLASS,
    ignore_filters=IGNORE_WHEN_FITTING,
    pca_batch_size=PCA_BATCH_SIZE,
    svm_c=SVM_C,
    random_state=RANDOM_STATE,
    max_plot_points_per_folder=MAX_PLOT_POINTS_PER_FOLDER,
    pca_filename=PCA_FILENAME,
    classifier_filename=CLASSIFIER_FILENAME,
    points_filename=POINTS_FILENAME,
    plot_filename=PLOT_FILENAME,
)
artifact_paths

Metadata cache: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\data\activation_explorer_cache
Ignored 10,268 rows; aggregated 201,975 fit rows into 4,784 points.
Balanced accuracy: 0.5620819397993311
              precision    recall  f1-score   support

     class_1       0.56      0.61      0.58 2391.9999999999995
     class_2       0.57      0.52      0.54 2391.9999999999995

    accuracy                           0.56 4783.999999999999
   macro avg       0.56      0.56      0.56 4783.999999999999
weighted avg       0.56      0.56      0.56 4783.999999999999



{'pca': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/pca_3_components_filter_2.joblib'),
 'classifier': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/balanced_linear_svm_filter_2.joblib'),
 'points': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/original_and_projected_pc_points_filter_2.csv'),
 'plot': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/results/balanced_pca_plane/pca_scores_and_plane_filter_2.html')}

## 2. Rewrite every activation row

Each row is independently transformed with the fitted PCA, orthogonally projected onto the fitted plane, and moved in activation space by the corresponding reconstruction difference. The output payload is identical to its source payload except for the replaced activation tensor.

In [3]:
rewrite_summary = rewrite_activation_batches(
    INPUT_ACTS_DIR,
    REWRITTEN_ACTS_DIR,
    pca_path=artifact_paths['pca'],
    classifier_path=artifact_paths['classifier'],
    folders=tuple(FOLDER_TO_CLASS),
    overwrite=OVERWRITE_REWRITTEN_BATCHES,
)
rewrite_summary

{'output_dir': WindowsPath('C:/Users/91967/Desktop/AISC/temporal-manifolds-last-position/.acts_filter_2'),
 'files': 1661,
 'rows': 212243,
 'maximum_plane_error': 1.1102230246251565e-16}

## Outputs

In [4]:
print('Model artifacts:')
for name, path in artifact_paths.items():
    print(f'  {name}: {path}')
print(f"Rewritten batches: {rewrite_summary['files']:,}")
print(f"Rewritten rows: {rewrite_summary['rows']:,}")
print(f"Activation output root: {rewrite_summary['output_dir']}")

Model artifacts:
  pca: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\pca_3_components_filter_2.joblib
  classifier: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\balanced_linear_svm_filter_2.joblib
  points: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\original_and_projected_pc_points_filter_2.csv
  plot: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\balanced_pca_plane\pca_scores_and_plane_filter_2.html
Rewritten batches: 1,661
Rewritten rows: 212,243
Activation output root: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\.acts_filter_2
